# 03 — Load TransLink Monthly Performance Data

## Performance Data — Auto-downloaded

These CSVs are automatically downloaded by `scripts/archive_gtfsrt.py` at startup
and refreshed every 24 hours. No manual steps needed.

If `source_files/performance/` is empty, run:
```
python scripts/archive_gtfsrt.py --download-performance-only
```

Or run the cell below to trigger a one-off download.

In [ ]:
# One-off download — reads URLs from config/feeds.yaml
import yaml
import requests
import time
from pathlib import Path

REPO_ROOT = Path.cwd().parent
with open(REPO_ROOT / 'config' / 'feeds.yaml') as f:
    config = yaml.safe_load(f)

output_dir = REPO_ROOT / 'source_files' / 'performance'
output_dir.mkdir(parents=True, exist_ok=True)

for key, meta in config.get('performance_csvs', {}).items():
    url = meta['url']
    filename = meta['filename']
    filepath = output_dir / filename
    try:
        r = requests.get(url, timeout=30)
        r.raise_for_status()
        filepath.write_bytes(r.content)
        rows = len(r.text.strip().splitlines()) - 1
        print(f'\u2713 {filename} \u2014 {rows} rows')
    except Exception as e:
        print(f'\u2717 {filename} \u2014 {e}')

In [ ]:
# Cell 2 — Load all CSVs from source_files/performance/ into a combined DataFrame
import pandas as pd
from pathlib import Path

REPO_ROOT = Path.cwd().parent
PERF_DIR = REPO_ROOT / 'source_files' / 'performance'

csv_files = sorted(PERF_DIR.glob('*.csv'))
if not csv_files:
    raise FileNotFoundError(
        f'No CSV files found in {PERF_DIR}.\n'
        'Please download them from the portal and re-run this cell.'
    )

print(f'Found {len(csv_files)} CSV file(s):')
for f in csv_files:
    print(f'  {f.name}')

dfs = [pd.read_csv(f) for f in csv_files]
raw = pd.concat(dfs, ignore_index=True)
print(f'\nCombined shape: {raw.shape}')
print('Columns:', list(raw.columns))
raw.head()

In [ ]:
# Cell 3 — Standardise column names
# TransLink CSV column names vary across releases — adjust mapping if needed.

COLUMN_MAP = {
    'Mode': 'mode', 'mode': 'mode',
    'Route': 'route', 'route': 'route', 'Route Number': 'route',
    'Month': 'month', 'month': 'month',
    'Year': 'year', 'year': 'year',
    'On Time Performance': 'on_time_pct', 'On Time %': 'on_time_pct',
    'OTP': 'on_time_pct', 'on_time_pct': 'on_time_pct',
    'Total Services': 'total_services', 'total_services': 'total_services',
    'Services': 'total_services',
}

df = raw.rename(columns=COLUMN_MAP)
KEEP = ['mode', 'route', 'month', 'year', 'on_time_pct', 'total_services']
df = df[[c for c in KEEP if c in df.columns]].copy()

if 'on_time_pct' in df.columns:
    df['on_time_pct'] = pd.to_numeric(
        df['on_time_pct'].astype(str).str.replace('%', '', regex=False), errors='coerce'
    )
if 'total_services' in df.columns:
    df['total_services'] = pd.to_numeric(df['total_services'], errors='coerce')
if 'year' in df.columns and 'month' in df.columns:
    df['date'] = pd.to_datetime(
        df['year'].astype(str) + '-' + df['month'].astype(str).str.zfill(2) + '-01',
        errors='coerce'
    )

print('Standardised schema:')
print(df.dtypes)
df.head()

In [ ]:
# Cell 4 — Summary: date range, modes, routes
print('Date range:', df['date'].min().strftime('%b %Y') if 'date' in df.columns else 'N/A',
      '\u2192', df['date'].max().strftime('%b %Y') if 'date' in df.columns else 'N/A')
print('Modes:', sorted(df['mode'].dropna().unique().tolist()) if 'mode' in df.columns else 'N/A')
print('Routes:', df['route'].nunique() if 'route' in df.columns else 'N/A')
print('Total rows:', len(df))
print('\nOn-time % summary:')
print(df['on_time_pct'].describe() if 'on_time_pct' in df.columns else 'column not found')

In [ ]:
# Cell 5 — Plot: overall on-time % by mode over time
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

if 'date' not in df.columns or 'mode' not in df.columns or 'on_time_pct' not in df.columns:
    print('Skipping plot — missing required columns (date, mode, on_time_pct)')
else:
    pivot = (
        df.groupby(['date', 'mode'])['on_time_pct']
        .mean()
        .unstack('mode')
    )

    fig, ax = plt.subplots(figsize=(12, 5))
    for mode in pivot.columns:
        ax.plot(pivot.index, pivot[mode], marker='o', markersize=3, label=mode)

    ax.set_title('SEQ TransLink On-Time Performance by Mode', fontsize=14, fontweight='bold')
    ax.set_xlabel('Month')
    ax.set_ylabel('On-Time %')
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%b %Y'))
    ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
    plt.xticks(rotation=45, ha='right')
    ax.legend(title='Mode')
    ax.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

In [ ]:
# Cell 6 — Save cleaned DataFrame as Parquet
out_path = PERF_DIR / 'performance_clean.parquet'
df.to_parquet(out_path, index=False)
print(f'Saved {len(df):,} rows \u2192 {out_path.relative_to(REPO_ROOT)}')
print('\nNext step: run notebooks/04_eda.ipynb')